## Analise do cenario TT LN brasil

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# ── Dados da tabela ───────────────────────────────────────────────────────────
regioes = ["MG", "SP", "NENO", "CO", "RJ", "SUL"]

jan = {
    "demanda":   [65538, 173637, 200754, 104493, 111219, 116261],
    "prod_real": [0,     256624, 143535, 100103, 267015, 81343],
    "prod_1w":   [0,     238579, 176297, 99590,  339790, 104891],
    "ei":        [28580, 114473, 136190, 61952,  84834,  86758],
    "suf_ini":   [11,    17,     15,     15,     20,     19],
    "transf":    [11586, -12424, 7456,   7620,  -29887,  12433],
    "efm":       [32771, 120086, 86427,  81962,  87765,  77470],
    "suf_f":     [11,    17,     12,     17,     16,     16],
}

fev = {
    "demanda":   [72849,  165262, 179674, 112802, 130649, 116585],
    "wsnp":      [0,      253602, 158000, 87686,  288283, 77748],
    "transf":    [75118,  -75583, 34507,  21794, -174401, 38222],
    "efm":       [35041,  132843, 99260,  78639,  70998,  76855],
    "suf_f":     [13,     26,     13,     21,     20,     17],
}

df_jan = pd.DataFrame(jan, index=regioes)
df_fev = pd.DataFrame(fev, index=regioes)

NENO_COLOR  = "#E24B4A"
OTHER_COLOR = "#B4B2A9"
BLUE        = "#378ADD"
GREEN       = "#1D9E75"
AMBER       = "#BA7517"
LAYOUT      = dict(template="plotly_white", font=dict(family="Arial", size=12))

def bar_color(regs, destaque="NENO", c1=NENO_COLOR, c2=OTHER_COLOR):
    return [c1 if r==destaque else c2 for r in regs]

# ── 1. Crescimento de demanda Jan→Fev ────────────────────────────────────────
delta_dem = [fev["demanda"][i] - jan["demanda"][i] for i in range(len(regioes))]
pct_dem   = [round((fev["demanda"][i]/jan["demanda"][i]-1)*100,1) for i in range(len(regioes))]

fig1 = make_subplots(rows=1, cols=2,
                     subplot_titles=["Demanda Jan vs Fev (hl)", "Variação % da demanda"])

fig1.add_trace(go.Bar(name="Janeiro", x=regioes, y=jan["demanda"],
                      marker_color=OTHER_COLOR, showlegend=True), row=1, col=1)
fig1.add_trace(go.Bar(name="Fevereiro", x=regioes, y=fev["demanda"],
                      marker_color=BLUE, showlegend=True), row=1, col=1)
fig1.add_trace(go.Bar(name="Variação %", x=regioes, y=pct_dem,
                      marker_color=bar_color(regioes),
                      text=[f"{v:+.1f}%" for v in pct_dem],
                      textposition="outside", showlegend=False), row=1, col=2)
fig1.add_hline(y=0, line_color="#333", line_width=0.5, row=1, col=2)

fig1.update_layout(**LAYOUT, title="Crescimento de demanda Long Neck — Jan → Fev",
                   barmode="group", height=440,
                   legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig1.update_yaxes(title_text="hl", row=1, col=1)
fig1.update_yaxes(title_text="%", row=1, col=2)
fig1.show()

# ── 2. NENO em foco: demanda vs produção Jan e Fev ───────────────────────────
fig2 = go.Figure()
categorias = ["Demanda Jan","Prod Jan (Real)","Prod Jan (1W)","Demanda Fev","Prod Fev (WSNP)"]
valores    = [jan["demanda"][2], jan["prod_real"][2], jan["prod_1w"][2],
              fev["demanda"][2], fev["wsnp"][2]]
cores      = [NENO_COLOR, BLUE, "#85B7EB", NENO_COLOR, BLUE]

fig2.add_trace(go.Bar(x=categorias, y=valores, marker_color=cores,
                      text=[f"{v:,.0f}" for v in valores], textposition="outside"))
fig2.add_hline(y=jan["demanda"][2], line_dash="dot", line_color=NENO_COLOR,
               annotation_text="demanda jan", annotation_position="top left")
fig2.add_hline(y=fev["demanda"][2], line_dash="dot", line_color=AMBER,
               annotation_text="demanda fev", annotation_position="top right")
fig2.update_layout(**LAYOUT, title="NENO — Demanda vs Produção: Jan e Fev",
                   height=440, yaxis_title="hl", showlegend=False,
                   yaxis_range=[0, max(valores)*1.2])
fig2.show()

# ── 3. Déficit de transferência de malha ─────────────────────────────────────
fig3 = make_subplots(rows=1, cols=2,
                     subplot_titles=["Transferência de malha — Janeiro",
                                     "Transferência de malha — Fevereiro"])
for col, (transf, titulo) in enumerate([(jan["transf"], "jan"), (fev["transf"], "fev")], start=1):
    cores = [GREEN if v>=0 else NENO_COLOR for v in transf]
    fig3.add_trace(go.Bar(
        x=regioes, y=transf,
        marker_color=cores,
        text=[f"{v:+,.0f}" for v in transf],
        textposition="outside",
        showlegend=False
    ), row=1, col=col)
    fig3.add_hline(y=0, line_color="#333", line_width=0.5, row=1, col=col)

fig3.update_layout(**LAYOUT,
                   title="Transferência de malha — positivo = recebe, negativo = envia (hl)",
                   height=440)
fig3.update_yaxes(title_text="hl")
fig3.add_annotation(text="NENO recebe volume das outras regiões para cobrir déficit de produção",
                    xref="paper", yref="paper", x=0.5, y=-0.15,
                    showarrow=False, font=dict(size=11, color="#888"), xanchor="center")
fig3.show()

# ── 4. Suficiência inicial vs final por região ───────────────────────────────
fig4 = make_subplots(rows=1, cols=2,
                     subplot_titles=["Suficiência — Janeiro", "Suficiência — Fevereiro"])

for col, (suf_i, suf_f, titulo) in enumerate([
    (jan["suf_ini"], jan["suf_f"], "jan"),
    ([None]*6,       fev["suf_f"], "fev")
], start=1):
    if col == 1:
        fig4.add_trace(go.Bar(name="Suf. inicial", x=regioes, y=suf_i,
                              marker_color="#B4B2A9", showlegend=True), row=1, col=col)
    fig4.add_trace(go.Bar(name="Suf. final" if col==1 else "Suf. final fev",
                          x=regioes, y=suf_f,
                          marker_color=bar_color(regioes, c1=NENO_COLOR, c2=BLUE),
                          text=suf_f, textposition="outside",
                          showlegend=True), row=1, col=col)
    fig4.add_hline(y=12, line_dash="dash", line_color=NENO_COLOR,
                   annotation_text="meta 12d", annotation_position="top right",
                   row=1, col=col)

fig4.update_layout(**LAYOUT, title="Suficiência por região — inicial vs final",
                   barmode="group", height=440,
                   legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig4.update_yaxes(title_text="Dias")
fig4.show()

# ── 5. Estoque final vs meta (waterfall NENO) ────────────────────────────────
# Decompõe o EFM do NENO em Jan: EI + Prod + Transf - Demanda
ei_neno   = jan["ei"][2]
prod_neno = jan["prod_real"][2]
trf_neno  = jan["transf"][2]
dem_neno  = jan["demanda"][2]
ef_neno   = jan["efm"][2]
meta_neno = 12 * (fev["demanda"][2] / 30)

fig5 = go.Figure(go.Waterfall(
    name="Composição EFM NENO — Janeiro",
    orientation="v",
    measure=["absolute","relative","relative","relative","total"],
    x=["Estoque Inicial","(+) Produção","(+) Recebimento malha","(–) Demanda","Estoque Final"],
    y=[ei_neno, prod_neno, trf_neno, -dem_neno, None],
    text=[f"{v:,.0f}" for v in [ei_neno, prod_neno, trf_neno, -dem_neno, ef_neno]],
    textposition="outside",
    connector=dict(line=dict(color="#ccc", width=1)),
    increasing=dict(marker=dict(color=GREEN)),
    decreasing=dict(marker=dict(color=NENO_COLOR)),
    totals=dict(marker=dict(color=BLUE)),
))
fig5.add_hline(y=meta_neno, line_dash="dash", line_color=AMBER,
               annotation_text=f"meta estoque ({meta_neno:,.0f} hl = 12d)",
               annotation_position="top right")
fig5.update_layout(**LAYOUT,
                   title="Waterfall — Composição do estoque final NENO em Janeiro",
                   height=480, yaxis_title="hl", showlegend=False)
fig5.show()

# ── 6. Ranking deficit: quem mais pressiona o sistema ────────────────────────
pressao_jan = [jan["demanda"][i] - jan["prod_real"][i] for i in range(len(regioes))]
pressao_fev = [fev["demanda"][i] - fev["wsnp"][i]     for i in range(len(regioes))]

fig6 = make_subplots(rows=1, cols=2,
                     subplot_titles=["Déficit produção vs demanda — Jan",
                                     "Déficit produção vs demanda — Fev"])
for col, pressao in enumerate([pressao_jan, pressao_fev], start=1):
    cores = [NENO_COLOR if v>0 else GREEN for v in pressao]
    fig6.add_trace(go.Bar(
        x=regioes, y=pressao,
        marker_color=cores,
        text=[f"{v:+,.0f}" for v in pressao],
        textposition="outside",
        showlegend=False
    ), row=1, col=col)
    fig6.add_hline(y=0, line_color="#333", line_width=0.8, row=1, col=col)

fig6.update_layout(**LAYOUT,
                   title="Déficit regional — Demanda menos Produção (hl) · positivo = deficit",
                   height=440)
fig6.update_yaxes(title_text="hl")
fig6.show()

## Data frame cenário com nova demanda 

In [2]:
import pandas as pd
from pyxlsb import open_workbook

FILE_PATH = r"excel\separado.por.cerveja.xlsb"

def get_bloco_dict(header, start, next_start):
    end = next_start if next_start else len(header)
    result = {}
    for i in range(start, end):
        val = str(header[i]).strip()
        if val not in ["", "None", "nan"]:
            result[val] = i
    return result

def ler_aba(sheet_name):
    rows = []
    with open_workbook(FILE_PATH) as wb:
        with wb.get_sheet(sheet_name) as sheet:
            for row in sheet.rows():
                rows.append([c.v for c in row])

    raw = pd.DataFrame(rows)

    header_row_idx = None
    for i, row in raw.iterrows():
        if row.astype(str).str.strip().str.lower().eq("demanda").any():
            header_row_idx = i
            break

    header = raw.iloc[header_row_idx].tolist()
    data   = raw.iloc[header_row_idx + 1:].reset_index(drop=True)

    semana_starts = [i for i, h in enumerate(header) if str(h).strip().lower() == "demanda"]
    COL_REGIAO    = next(i for i, h in enumerate(header) if str(h).strip() not in ["", "None", "nan"])

    records = []
    for idx_semana, col_start in enumerate(semana_starts):
        next_start = semana_starts[idx_semana + 1] if idx_semana + 1 < len(semana_starts) else None
        bloco = get_bloco_dict(header, col_start, next_start)

        def get_col(row, nomes):
            for n in nomes:
                if n in bloco:
                    return pd.to_numeric(row.iloc[bloco[n]], errors="coerce")
            return None

        for _, row in data.iterrows():
            regiao = row.iloc[COL_REGIAO]
            if pd.isna(regiao) or str(regiao).strip() in ["", "None", "TOTAL"]:
                continue

            records.append({
                "Cerveja":       sheet_name,
                "Regiao":        str(regiao).strip(),
                "Semana":        f"W{idx_semana}",
                "Demanda":       get_col(row, ["Demanda"]),
                "WSNP":          get_col(row, ["WSNP", "Malha", "0.0"]),
                "EI_Semana":     get_col(row, ["EI Semana"]),
                "Suf_in":        get_col(row, ["Suf.ini(d)"]),
                "Transi_Intern": get_col(row, ["Transf. Interna"]),
                "Transi_Cabo":   get_col(row, ["Transf. Ext (Cabo)"]),
                "Transi_Rodo":   get_col(row, ["Transf. Ext (Rodo)"]),
                "Transito":      get_col(row, ["Trânsito"]),
                "EF_Semana":     get_col(row, ["EF Semana"]),
                "Suf_f":         get_col(row, ["Suf. f (d)"]),
            })

    return pd.DataFrame(records)

# ── Ler todas as abas ─────────────────────────────────────────────────────────
abas = ["patagonia", "goose", "malzbier ", "colorado "]
df_all = pd.concat([ler_aba(a) for a in abas], ignore_index=True)

# ── Exibir por cerveja e semana ───────────────────────────────────────────────
for cerveja, df_cerv in df_all.groupby("Cerveja"):
    print(f"\n{'#'*60}")
    print(f"  {cerveja.upper()}")
    print(f"{'#'*60}")
    for semana, grupo in df_cerv.groupby("Semana"):
        print(f"\n{'='*60}  {semana}  {'='*60}")
        display(grupo.reset_index(drop=True))
ei_suf_w0 = df_all[df_all["Semana"] == "W0"][["Cerveja", "Regiao", "EI_Semana", "Suf_in"]]




############################################################
  COLORADO 
############################################################

============================================================  W0  ============================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,EI_Semana,Suf_in,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f
0,colorado,Mapapi,W0,1854.198,0.0,3877.200,12.546233,2891.44836,0.0,0.0,0.0,4914.45036,15.117385
1,colorado,NE Norte,W0,1370.880,5400.0,8016.278,35.085250,-4225.04262,0.0,0.0,0.0,7820.35538,28.970395
2,colorado,NE Sul,W0,1199.718,0.0,5080.104,25.406491,228.44160,0.0,0.0,0.0,4108.82760,17.807484
3,colorado,NO Araguaia,W0,48.114,0.0,0.000,0.000000,48.11058,0.0,0.0,0.0,-0.00342,-0.000357
4,colorado,NO Centro,W0,1579.266,0.0,4416.696,16.780059,1056.66468,0.0,0.0,0.0,3894.09468,13.736801



============================================================  W1  ============================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,EI_Semana,Suf_in,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f
0,colorado,Mapapi,W1,1950.516,0.0,NaN,NaN,1112.50674,0.0,0.0,0.0,4076.44110,17.169529
1,colorado,NE Norte,W1,1619.658,0.0,NaN,NaN,-3080.02788,0.0,0.0,0.0,3120.66950,14.772120
2,colorado,NE Sul,W1,1384.416,0.0,NaN,NaN,525.88278,0.0,0.0,0.0,3250.29438,18.542066
3,colorado,NO Araguaia,W1,57.474,0.0,NaN,NaN,38.14938,0.0,0.0,0.0,-19.32804,-2.977209
4,colorado,NO Centro,W1,1700.874,0.0,NaN,NaN,1403.23998,-228.0,0.0,0.0,3368.46066,16.545396



============================================================  W2  ============================================================


...

,Cerveja,Regiao,Semana,Demanda,WSNP,EI_Semana,Suf_in,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f
0,patagonia,Mapapi,W1,2914.344,0.0,NaN,NaN,2186.21268,0.0,0.0,0.0,3214.59336,8.920728
1,patagonia,NE Norte,W1,2494.458,0.0,NaN,NaN,1831.99770,0.0,0.0,0.0,4151.16072,13.666778
2,patagonia,NE Sul,W1,2160.486,0.0,NaN,NaN,2893.89996,0.0,0.0,0.0,5045.89392,19.356956
3,patagonia,NO Araguaia,W1,135.954,0.0,NaN,NaN,135.95562,0.0,0.0,0.0,0.00918,0.000598
4,patagonia,NO Centro,W1,1543.284,1800.0,NaN,NaN,-7047.86434,-515.0,0.0,0.0,-186.01770,-0.960364



============================================================  W2  ============================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,EI_Semana,Suf_in,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f
0,patagonia,Mapapi,W2,2162.106,0.0,NaN,NaN,1726.37370,0.0,0.0,0.0,2778.86106,7.150366
1,patagonia,NE Norte,W2,1822.446,0.0,NaN,NaN,1735.92756,0.0,0.0,0.0,4064.64228,12.151179
2,patagonia,NE Sul,W2,1564.056,0.0,NaN,NaN,376.32960,0.0,0.0,0.0,3858.16752,13.378716
3,patagonia,NO Araguaia,W2,92.106,0.0,NaN,NaN,92.09898,0.0,0.0,0.0,0.00216,0.000134
4,patagonia,NO Centro,W2,1162.170,5040.0,NaN,NaN,-3930.65812,-35.0,0.0,0.0,-273.84582,-1.272772



============================================================  W3  ============================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,EI_Semana,Suf_in,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f
0,patagonia,Mapapi,W3,2331.792,0.0,NaN,NaN,3298.08654,0.0,0.0,0.0,3745.15560,9.636766
1,patagonia,NE Norte,W3,2007.036,0.0,NaN,NaN,1695.43332,0.0,0.0,0.0,3753.03960,11.219648
2,patagonia,NE Sul,W3,1730.286,0.0,NaN,NaN,5200.78914,0.0,0.0,0.0,7328.67066,25.413154
3,patagonia,NO Araguaia,W3,96.498,0.0,NaN,NaN,96.48468,0.0,0.0,0.0,-0.01116,-0.000694
4,patagonia,NO Centro,W3,1290.942,12600.0,NaN,NaN,-10290.65986,-1088.0,0.0,0.0,-343.44768,-1.596265


## Redistribuicao producao no neno. 
### considera transferencia externa por cabotagem 

In [3]:
producao = {
    "patagonia": {"CE": {"W0": 12240, "W1": 1800,  "W2": 5040,  "W3": 12600},
                  "PE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0}},
    "malzbier ": {"CE": {"W0": 0,     "W1": 9000,   "W2": 7560,  "W3": 0},
                  "PE": {"W0": 16200, "W1": 0,      "W2": 12960, "W3": 0}},
    "colorado ": {"CE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0},
                  "PE": {"W0": 5400,  "W1": 0,      "W2": 10800, "W3": 0}},
    "goose":     {"CE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0},
                  "PE": {"W0": 5400,  "W1": 14400,  "W2": 0,     "W3": 12600}},
}


def analisar_cerveja(cerveja, df_all, producao):
    df_cerv = df_all[df_all["Cerveja"] == cerveja].copy()
    ei_w0   = df_cerv[df_cerv["Semana"] == "W0"].set_index("Regiao")["EI_Semana"].to_dict()
    semanas = ["W0", "W1", "W2", "W3"]
    regioes = sorted(df_cerv["Regiao"].unique().tolist())

    dem_dict  = {}
    cabo_dict = {}
    for sem in semanas:
        sub = df_cerv[df_cerv["Semana"] == sem].set_index("Regiao")
        dem_dict[sem]  = sub["Demanda"].to_dict()
        cabo_dict[sem] = sub["Transi_Cabo"].fillna(0).to_dict()

    estoque    = {r: float(ei_w0.get(r, 0) or 0) for r in regioes}
    resultados = []

    for i, sem in enumerate(semanas):
        prod_disp = producao[cerveja]["CE"].get(sem, 0) + producao[cerveja]["PE"].get(sem, 0)
        prox_sem  = semanas[i + 1] if i + 1 < len(semanas) else None

        nec = {}
        for r in regioes:
            dem      = dem_dict[sem].get(r, 0)
            ei       = estoque[r]
            t_cabo   = cabo_dict[sem].get(r, 0)
            dem_prox = dem_dict[prox_sem].get(r, 0) if prox_sem else dem
            ef_alvo  = 12 * (dem_prox / 6)
            ef_base  = ei + t_cabo - dem
            envio_nec = max(0, ef_alvo - ef_base)
            nec[r] = {"dem": dem, "ei": ei, "t_cabo": t_cabo,
                      "ef_alvo": ef_alvo, "envio_nec": envio_nec, "dem_prox": dem_prox}

        total_nec = sum(v["envio_nec"] for v in nec.values())

        for r in regioes:
            dem       = nec[r]["dem"]
            ei        = nec[r]["ei"]
            t_cabo    = nec[r]["t_cabo"]
            ef_alvo   = nec[r]["ef_alvo"]
            envio_nec = nec[r]["envio_nec"]
            dem_prox  = nec[r]["dem_prox"]

            envio_prod = prod_disp * (envio_nec / total_nec) if total_nec > 0 else 0
            vem_sp     = max(0, envio_nec - envio_prod)

            if sem in ["W0", "W1", "W2"]:
                modal         = "Rodoviário"
                vol_enviar_sp = vem_sp / 0.95
            else:
                modal         = "Cabotagem"
                vol_enviar_sp = vem_sp

            ef    = ei + envio_prod + vem_sp + t_cabo - dem
            suf_f = ef / (dem_prox / 6) if dem_prox > 0 else 0

            resultados.append({
                "Cerveja":       cerveja,
                "Semana":        sem,
                "Regiao":        r,
                "Demanda":       round(dem, 2),
                "Dem_Proxima":   round(dem_prox, 2),
                "EI":            round(ei, 2),
                "Transi_Cabo":   round(t_cabo, 2),
                "Prod_NE":       round(envio_prod, 2),
                "Vem_SP":        round(vem_sp, 2),
                "Vol_Enviar_SP": round(vol_enviar_sp, 2),
                "Modal_SP":      modal if vem_sp > 0 else "-",
                "EF":            round(ef, 2),
                "Suf_f":         round(suf_f, 1),
            })
            estoque[r] = ef

    return pd.DataFrame(resultados)

cervejas = ["patagonia", "malzbier ", "colorado ", "goose"]
df_final = pd.concat([analisar_cerveja(c, df_all, producao) for c in cervejas], ignore_index=True)

for cerveja, df_c in df_final.groupby("Cerveja"):
    print(f"\n{'#'*60}  {cerveja.upper()}  {'#'*60}")
    for sem, grupo in df_c.groupby("Semana"):
        print(f"\n{'='*50}  {sem}  {'='*50}")
        display(grupo.reset_index(drop=True))

print("\n=== RESUMO — O QUE VEM DE SP ===")
sp = df_final[df_final["Vem_SP"] > 0][
    ["Cerveja","Semana","Regiao","Demanda","Transi_Cabo","Prod_NE","Vem_SP","Vol_Enviar_SP","Modal_SP","Suf_f"]]
display(sp if len(sp) > 0 else pd.DataFrame([{"Resultado": "Nenhuma transferência necessária!"}]))

print("\n=== PRODUÇÃO TOTAL NE POR CERVEJA/SEMANA ===")
display(df_final.groupby(["Cerveja","Semana"])["Prod_NE"].sum().reset_index())


############################################################  COLORADO   ############################################################

==================================================  W0  ==================================================


,Cerveja,Semana,Regiao,Demanda,Dem_Proxima,EI,Transi_Cabo,Prod_NE,Vem_SP,Vol_Enviar_SP,Modal_SP,EF,Suf_f
0,colorado,W0,Mapapi,1854.20,1950.52,3877.20,0.0,3892.42,0.0,0.0,-,5915.43,18.2
1,colorado,W0,NE Norte,1370.88,1619.66,8016.28,0.0,0.00,0.0,0.0,-,6645.40,24.6
2,colorado,W0,NE Sul,1199.72,1384.42,5080.10,0.0,0.00,0.0,0.0,-,3880.39,16.8
3,colorado,W0,NO Araguaia,48.11,57.47,0.00,0.0,337.96,0.0,0.0,-,289.85,30.3
4,colorado,W0,NO Centro,1579.27,1700.87,4416.70,0.0,1169.61,0.0,0.0,-,4007.04,14.1



==================================================  W1  ==================================================


,Cerveja,Semana,Regiao,Demanda,Dem_Proxima,EI,Transi_Cabo,Prod_NE,Vem_SP,Vol_Enviar_SP,Modal_SP,EF,Suf_f
0,colorado,W1,Mapapi,1950.52,1424.54,5915.43,0.0,0.0,0.0,0.00,-,3964.91,16.7
1,colorado,W1,NE Norte,1619.66,1267.52,6645.40,0.0,0.0,0.0,0.00,-,5025.74,23.8
2,colorado,W1,NE Sul,1384.42,1051.76,3880.39,0.0,0.0,0.0,0.00,-,2495.97,14.2
3,colorado,W1,NO Araguaia,57.47,38.95,289.85,0.0,0.0,0.0,0.00,-,232.38,35.8
4,colorado,W1,NO Centro,1700.87,1221.53,4007.04,-228.0,0.0,364.9,384.11,Rodoviário,2443.07,12.0



==================================================  W2  ==================================================


...

,Cerveja,Semana,Regiao,Demanda,Dem_Proxima,EI,Transi_Cabo,Prod_NE,Vem_SP,Vol_Enviar_SP,Modal_SP,EF,Suf_f
0,patagonia,W3,Mapapi,2331.79,2331.79,4663.58,0.0,3438.52,0.0,0.0,-,5770.31,14.8
1,patagonia,W3,NE Norte,2007.04,2007.04,4014.07,0.0,2959.62,0.0,0.0,-,4966.66,14.8
2,patagonia,W3,NE Sul,1730.29,1730.29,3460.57,0.0,2551.52,0.0,0.0,-,4281.81,14.8
3,patagonia,W3,NO Araguaia,96.50,96.50,193.00,0.0,142.30,0.0,0.0,-,238.80,14.8
4,patagonia,W3,NO Centro,1290.94,1290.94,2581.88,-1088.0,3508.04,0.0,0.0,-,3710.99,17.2



=== RESUMO — O QUE VEM DE SP ===


,Cerveja,Semana,Regiao,Demanda,Transi_Cabo,Prod_NE,Vem_SP,Vol_Enviar_SP,Modal_SP,Suf_f
5,patagonia,W1,Mapapi,2914.34,0.0,371.30,287.49,302.62,Rodoviário,12.0
6,patagonia,W1,NE Norte,2494.46,0.0,505.63,391.49,412.10,Rodoviário,12.0
7,patagonia,W1,NE Sul,2160.49,0.0,461.85,357.59,376.41,Rodoviário,12.0
9,patagonia,W1,NO Centro,1543.28,-515.0,461.21,357.10,375.89,Rodoviário,12.0
10,patagonia,W2,Mapapi,2162.11,0.0,1550.44,951.04,1001.09,Rodoviário,12.0
11,patagonia,W2,NE Norte,1822.45,0.0,1358.39,833.24,877.09,Rodoviário,12.0
12,patagonia,W2,NE Sul,1564.06,0.0,1175.48,721.04,758.99,Rodoviário,12.0
13,patagonia,W2,NO Araguaia,92.11,0.0,54.05,33.15,34.90,Rodoviário,12.0
14,patagonia,W2,NO Centro,1162.17,-35.0,901.64,553.07,582.18,Rodoviário,12.0
20,malzbier,W0,Mapapi,6074.80,0.0,7668.44,8993.56,9466.91,Rodoviário,12.0



=== PRODUÇÃO TOTAL NE POR CERVEJA/SEMANA ===


,Cerveja,Semana,Prod_NE
0,colorado,W0,5399.99
1,colorado,W1,0.00
2,colorado,W2,10800.00
3,colorado,W3,0.00
4,goose,W0,5400.00
5,goose,W1,14400.00
6,goose,W2,0.00
7,goose,W3,12600.00
8,malzbier,W0,16199.99
9,malzbier,W1,9000.00


## Maximizar producao - O que produzir?

#### capacidade de aumentar 360 na w0 - malzbier (muita demanda na semana 0, frete mais caro)
#### capacidade de aumentar 1800 + 7200 na semana w1

In [4]:
# ── Fatores de custo ──────────────────────────────────────────────────────────
custos = {
    "patagonia": {"maco": 285, "custo_prod": 149, "frete_cabo": 0,  "frete_rodo": 0},
    "malzbier":  {"maco": 285, "custo_prod": 149, "frete_cabo": 85, "frete_rodo": 85 * 1.6},
    "colorado":  {"maco": 300, "custo_prod": 150, "frete_cabo": 77, "frete_rodo": 77 * 1.6},
    "goose":     {"maco": 350, "custo_prod": 155, "frete_cabo": 85, "frete_rodo": 85 * 1.6},
}

AVARIA_RODO = 0.05
AVARIA_CABO = 0.00

def custo_entregue(cerveja, modal, origem="NE"):
    c = custos[cerveja]
    if origem == "NE":
        return round(c["custo_prod"], 2)
    else:
        if modal == "Rodoviário":
            frete  = c["frete_rodo"]
            avaria = AVARIA_RODO
        else:
            frete  = c["frete_cabo"]
            avaria = AVARIA_CABO
        vol_enviar = 1 / (1 - avaria)
        return round(c["custo_prod"] * vol_enviar + frete, 2)

def margem_entregue(cerveja, modal, origem="NE"):
    return round(custos[cerveja]["maco"] - custo_entregue(cerveja, modal, origem), 2)# ── Capacidade por fábrica ────────────────────────────────────────────────────
# CE: pode produzir malzbier, patagonia, colorado
# PE: pode produzir malzbier, colorado (goose FIXO — não mexer)

opcoes_ce = ["malzbier", "patagonia", "colorado"]
opcoes_pe = ["malzbier", "colorado"]

print("=" * 80)
print("ANÁLISE: o que produzir com os extras na W1")
print("W1 CE: +1800 hl | W1 PE: +7200 hl")
print("Referência: W0/W1/W2 = rodoviário (o que não produzir no NE vem de SP assim)")
print("=" * 80)

print("\n▶ W1 CE — +1800 hl — opções: Malzbier, Patagonia, Colorado")
for c in opcoes_ce:
    mg_ne = margem_entregue(c, "-", "NE")
    mg_sp = margem_entregue(c, "Rodoviário", "SP")
    ganho = 1800 * (mg_ne - mg_sp)
    print(f"  {c.upper():12} → Margem NE: R${mg_ne:>7.2f}/hl  |  SP Rodo: R${mg_sp:>7.2f}/hl  |  Ganho total: R${ganho:>10,.2f}")

print("\n▶ W1 PE — +7200 hl — opções: Malzbier, Colorado")
for c in opcoes_pe:
    mg_ne = margem_entregue(c, "-", "NE")
    mg_sp = margem_entregue(c, "Rodoviário", "SP")
    ganho = 7200 * (mg_ne - mg_sp)
    print(f"  {c.upper():12} → Margem NE: R${mg_ne:>7.2f}/hl  |  SP Rodo: R${mg_sp:>7.2f}/hl  |  Ganho total: R${ganho:>10,.2f}")

print("\n✅ RECOMENDAÇÃO:")
best_ce = max(opcoes_ce, key=lambda c: margem_entregue(c,"-","NE") - margem_entregue(c,"Rodoviário","SP"))
best_pe = max(opcoes_pe, key=lambda c: margem_entregue(c,"-","NE") - margem_entregue(c,"Rodoviário","SP"))
print(f"  CE W1 +1800 hl → {best_ce.upper()} (maior ganho vs trazer de SP)")
print(f"  PE W1 +7200 hl → {best_pe.upper()} (maior ganho vs trazer de SP)")

ANÁLISE: o que produzir com os extras na W1
W1 CE: +1800 hl | W1 PE: +7200 hl
Referência: W0/W1/W2 = rodoviário (o que não produzir no NE vem de SP assim)

▶ W1 CE — +1800 hl — opções: Malzbier, Patagonia, Colorado
  MALZBIER     → Margem NE: R$ 136.00/hl  |  SP Rodo: R$  -7.84/hl  |  Ganho total: R$258,912.00
  PATAGONIA    → Margem NE: R$ 136.00/hl  |  SP Rodo: R$ 128.16/hl  |  Ganho total: R$ 14,112.00
  COLORADO     → Margem NE: R$ 150.00/hl  |  SP Rodo: R$  18.91/hl  |  Ganho total: R$235,962.00

▶ W1 PE — +7200 hl — opções: Malzbier, Colorado
  MALZBIER     → Margem NE: R$ 136.00/hl  |  SP Rodo: R$  -7.84/hl  |  Ganho total: R$1,035,648.00
  COLORADO     → Margem NE: R$ 150.00/hl  |  SP Rodo: R$  18.91/hl  |  Ganho total: R$943,848.00

✅ RECOMENDAÇÃO:
  CE W1 +1800 hl → MALZBIER (maior ganho vs trazer de SP)
  PE W1 +7200 hl → MALZBIER (maior ganho vs trazer de SP)


In [8]:
# ── Fatores de custo ──────────────────────────────────────────────────────────
custos = {
    "patagonia": {"maco": 285, "custo_prod": 149, "frete_cabo": 0,  "frete_rodo": 0},
    "malzbier":  {"maco": 285, "custo_prod": 149, "frete_cabo": 85, "frete_rodo": 85 * 1.6},
    "colorado":  {"maco": 300, "custo_prod": 150, "frete_cabo": 77, "frete_rodo": 77 * 1.6},
    "goose":     {"maco": 350, "custo_prod": 155, "frete_cabo": 85, "frete_rodo": 85 * 1.6},
}

AVARIA_RODO = 0.05
AVARIA_CABO = 0.00

def custo_entregue(cerveja, modal, origem="NE"):
    c = custos[cerveja]
    if origem == "NE":
        return round(c["custo_prod"], 2)
    else:
        if modal == "Rodoviário":
            frete  = c["frete_rodo"]
            avaria = AVARIA_RODO
        else:
            frete  = c["frete_cabo"]
            avaria = AVARIA_CABO
        vol_enviar = 1 / (1 - avaria)
        return round(c["custo_prod"] * vol_enviar + frete, 2)

def margem_entregue(cerveja, modal, origem="NE"):
    return round(custos[cerveja]["maco"] - custo_entregue(cerveja, modal, origem), 2)

# ── Capacidade por fábrica ────────────────────────────────────────────────────
# CE: malzbier, patagonia, colorado
# PE: malzbier, colorado (goose FIXO)
opcoes_ce = ["malzbier", "patagonia", "colorado"]
opcoes_pe = ["malzbier", "colorado"]

print("=" * 80)
print("ANÁLISE: o que produzir com os extras na W1")
print("W1 CE: +1800 hl | W1 PE: +7200 hl")
print("Referência: W0/W1/W2 = rodoviário")
print("=" * 80)

print("\n▶ W1 CE — +1800 hl — opções: Malzbier, Patagonia, Colorado")
for c in opcoes_ce:
    mg_ne = margem_entregue(c, "-",          "NE")
    mg_sp = margem_entregue(c, "Rodoviário", "SP")
    ganho = 1800 * (mg_ne - mg_sp)
    print(f"  {c.upper():12} → Margem NE: R${mg_ne:>7.2f}/hl  |  SP Rodo: R${mg_sp:>7.2f}/hl  |  Ganho total: R${ganho:>10,.2f}")

print("\n▶ W1 PE — +7200 hl — opções: Malzbier, Colorado")
for c in opcoes_pe:
    mg_ne = margem_entregue(c, "-",          "NE")
    mg_sp = margem_entregue(c, "Rodoviário", "SP")
    ganho = 7200 * (mg_ne - mg_sp)
    print(f"  {c.upper():12} → Margem NE: R${mg_ne:>7.2f}/hl  |  SP Rodo: R${mg_sp:>7.2f}/hl  |  Ganho total: R${ganho:>10,.2f}")

best_ce = max(opcoes_ce, key=lambda c: margem_entregue(c,"-","NE") - margem_entregue(c,"Rodoviário","SP"))
best_pe = max(opcoes_pe, key=lambda c: margem_entregue(c,"-","NE") - margem_entregue(c,"Rodoviário","SP"))

print(f"\n✅ RECOMENDAÇÃO:")
print(f"  CE W1 +1800 hl → {best_ce.upper()}")
print(f"  PE W1 +7200 hl → {best_pe.upper()}")

# ── Bibliotecas de produção ───────────────────────────────────────────────────
producao_base = {
    "patagonia": {"CE": {"W0": 12240, "W1": 1800,  "W2": 5040,  "W3": 12600},
                  "PE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0}},
    "malzbier ": {"CE": {"W0": 0,     "W1": 9000,   "W2": 7560,  "W3": 0},
                  "PE": {"W0": 16200, "W1": 0,      "W2": 12960, "W3": 0}},
    "colorado ": {"CE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0},
                  "PE": {"W0": 5400,  "W1": 0,      "W2": 10800, "W3": 0}},
    "goose":     {"CE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0},
                  "PE": {"W0": 5400,  "W1": 14400,  "W2": 0,     "W3": 12600}},
}

# W0 CE: +360 malzbier
# W1 CE: +1800 → best_ce
# W1 PE: +7200 → best_pe
def monta_producao_otim(best_ce, best_pe):
    import copy
    otim = copy.deepcopy(producao_base)

    # W0 CE: +360 malzbier
    otim["malzbier "]["CE"]["W0"] += 360

    # W1 CE: +1800 na cerveja recomendada
    chave_ce = best_ce + " " if best_ce in ["malzbier", "colorado"] else best_ce
    if chave_ce not in otim:
        chave_ce = best_ce
    otim[chave_ce]["CE"]["W1"] += 1800

    # W1 PE: +7200 na cerveja recomendada
    chave_pe = best_pe + " " if best_pe in ["malzbier", "colorado"] else best_pe
    if chave_pe not in otim:
        chave_pe = best_pe
    otim[chave_pe]["PE"]["W1"] += 7200

    return otim

producao_otim = monta_producao_otim(best_ce, best_pe)

print("\n" + "=" * 80)
print("PRODUÇÃO BASE vs OTIMIZADA — total por cerveja")
print("=" * 80)
for c in producao_base:
    t_base = sum(v for f in producao_base[c].values() for v in f.values())
    t_otim = sum(v for f in producao_otim[c].values() for v in f.values())
    diff   = t_otim - t_base
    sinal  = f"+{diff:,.0f}" if diff >= 0 else f"{diff:,.0f}"
    print(f"  {c.strip():12} Base: {t_base:>8,.0f} hl  |  Otim: {t_otim:>8,.0f} hl  |  Δ {sinal} hl")

# ── Roda redistribuição com produção otimizada ────────────────────────────────
cervejas = ["patagonia", "malzbier ", "colorado ", "goose"]

df_otim = pd.concat(
    [analisar_cerveja(c, df_all, producao_otim) for c in cervejas],
    ignore_index=True
)

# ── Custo total: base vs otimizado ────────────────────────────────────────────
def calc_custo_total(df_use):
    total = 0
    for _, row in df_use.iterrows():
        c = row["Cerveja"].strip()
        if c not in custos:
            continue
        modal = row["Modal_SP"] if row["Modal_SP"] != "-" else "Rodoviário"
        total += custo_entregue(c, "-",   "NE") * row["Prod_NE"]
        total += custo_entregue(c, modal, "SP") * row["Vem_SP"]
    return total

def calc_receita(df_use):
    total = 0
    for _, row in df_use.iterrows():
        c = row["Cerveja"].strip()
        if c not in custos:
            continue
        total += custos[c]["maco"] * row["Demanda"]
    return total

print("\n" + "=" * 80)
print("RESULTADO FINANCEIRO — BASE vs OTIMIZADO")
print("=" * 80)
for label, df_use in [("Base",      df_final),
                       ("Otimizado", df_otim)]:
    custo   = calc_custo_total(df_use)
    receita = calc_receita(df_use)
    margem  = receita - custo
    print(f"\n  {label}")
    print(f"    Custo produção + frete: R$ {custo:>15,.2f}")
    print(f"    Receita (MACO):         R$ {receita:>15,.2f}")
    print(f"    Margem líquida:         R$ {margem:>15,.2f}")

ganho_otim = calc_receita(df_otim) - calc_custo_total(df_otim) - \
             (calc_receita(df_final) - calc_custo_total(df_final))
print(f"\n  💰 Ganho incremental otimização: R$ {ganho_otim:,.2f}")

# ── Resumo SP: base vs otimizado ─────────────────────────────────────────────
print("\n" + "=" * 80)
print("VOLUME DE SP — BASE vs OTIMIZADO")
print("=" * 80)
for label, df_use in [("Base", df_final), ("Otimizado", df_otim)]:
    sp = df_use.groupby(["Cerveja","Semana"])[["Vem_SP","Vol_Enviar_SP"]].sum()
    sp = sp[sp["Vem_SP"] > 0]
    print(f"\n  {label}")
    display(sp.reset_index())

ANÁLISE: o que produzir com os extras na W1
W1 CE: +1800 hl | W1 PE: +7200 hl
Referência: W0/W1/W2 = rodoviário

▶ W1 CE — +1800 hl — opções: Malzbier, Patagonia, Colorado
  MALZBIER     → Margem NE: R$ 136.00/hl  |  SP Rodo: R$  -7.84/hl  |  Ganho total: R$258,912.00
  PATAGONIA    → Margem NE: R$ 136.00/hl  |  SP Rodo: R$ 128.16/hl  |  Ganho total: R$ 14,112.00
  COLORADO     → Margem NE: R$ 150.00/hl  |  SP Rodo: R$  18.91/hl  |  Ganho total: R$235,962.00

▶ W1 PE — +7200 hl — opções: Malzbier, Colorado
  MALZBIER     → Margem NE: R$ 136.00/hl  |  SP Rodo: R$  -7.84/hl  |  Ganho total: R$1,035,648.00
  COLORADO     → Margem NE: R$ 150.00/hl  |  SP Rodo: R$  18.91/hl  |  Ganho total: R$943,848.00

✅ RECOMENDAÇÃO:
  CE W1 +1800 hl → MALZBIER
  PE W1 +7200 hl → MALZBIER

PRODUÇÃO BASE vs OTIMIZADA — total por cerveja
  patagonia    Base:   31,680 hl  |  Otim:   31,680 hl  |  Δ +0 hl
  malzbier     Base:   45,720 hl  |  Otim:   55,080 hl  |  Δ +9,360 hl
  colorado     Base:   16,200 hl 

,Cerveja,Semana,Vem_SP,Vol_Enviar_SP
0,colorado,W1,364.90,384.11
1,colorado,W3,533.55,533.55
2,goose,W0,10471.39,11022.52
3,malzbier,W0,18999.39,19999.35
4,malzbier,W3,2254.72,2254.72
5,patagonia,W1,1393.67,1467.02
6,patagonia,W2,3091.54,3254.25



  Otimizado


,Cerveja,Semana,Vem_SP,Vol_Enviar_SP
0,colorado,W1,364.90,384.11
1,colorado,W3,533.55,533.55
2,goose,W0,10471.39,11022.52
3,malzbier,W0,18639.38,19620.40
4,malzbier,W3,5278.51,5278.51
5,patagonia,W1,1393.67,1467.02
6,patagonia,W2,3091.54,3254.25


In [9]:
cervejas = ["patagonia", "malzbier ", "colorado ", "goose"]

df_otim = pd.concat(
    [analisar_cerveja(c, df_all, producao_otim) for c in cervejas],
    ignore_index=True
)

print("✅ df_otim gerado!")
print(f"   Shape: {df_otim.shape}")
print(f"\n=== PRODUÇÃO NE TOTAL POR CERVEJA/SEMANA ===")
display(df_otim.groupby(["Cerveja","Semana"])[["Prod_NE","Vem_SP","Vol_Enviar_SP"]].sum().reset_index())

print(f"\n=== SUFICIÊNCIA FINAL ===")
display(df_otim.groupby(["Cerveja","Semana","Regiao"])["Suf_f"].first().reset_index())

✅ df_otim gerado!
   Shape: (80, 13)

=== PRODUÇÃO NE TOTAL POR CERVEJA/SEMANA ===


,Cerveja,Semana,Prod_NE,Vem_SP,Vol_Enviar_SP
0,colorado,W0,5399.99,0.00,0.00
1,colorado,W1,0.00,364.90,384.11
2,colorado,W2,10800.00,0.00,0.00
3,colorado,W3,0.00,533.55,533.55
4,goose,W0,5400.00,10471.39,11022.52
5,goose,W1,14400.00,0.00,0.00
6,goose,W2,0.00,0.00,0.00
7,goose,W3,12600.00,0.00,0.00
8,malzbier,W0,16559.99,18639.38,19620.40
9,malzbier,W1,18000.01,0.00,0.00



=== SUFICIÊNCIA FINAL ===


,Cerveja,Semana,Regiao,Suf_f
0,colorado,W0,Mapapi,18.2
1,colorado,W0,NE Norte,24.6
2,colorado,W0,NE Sul,16.8
3,colorado,W0,NO Araguaia,30.3
4,colorado,W0,NO Centro,14.1
...,...,...,...,...
75,patagonia,W3,Mapapi,14.8
76,patagonia,W3,NE Norte,14.8
77,patagonia,W3,NE Sul,14.8
78,patagonia,W3,NO Araguaia,14.8


In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import plotly.io as pio
pio.renderers.default = "browser"
import plotly.io as pio
pio.renderers.default = "vscode"

regioes    = ["Mapapi", "NE Norte", "NE Sul", "NO Araguaia", "NO Centro"]
cervejas   = ["patagonia", "malzbier ", "colorado ", "goose"]
semanas    = ["W0", "W1", "W2", "W3"]
cervejas_k = [c.strip() for c in cervejas]

LAYOUT     = dict(template="plotly_white", font=dict(family="Arial", size=12))
COLORSCALE = [[0.0,"#E24B4A"],[0.25,"#F09595"],[0.45,"#FAC775"],
              [0.5,"#FAC775"],[0.55,"#85B7EB"],[0.75,"#378ADD"],[1.0,"#1D9E75"]]
COR_CERV   = {"patagonia":"#378ADD","malzbier":"#1D9E75","colorado":"#BA7517","goose":"#D4537E"}

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 1 — Suficiência SEM o que vem de SP
# (efeito PURO da redistribuição da produção NE)
# EF_redist = EI + Prod_NE + Transi_Cabo - Demanda   (sem Vem_SP)
# ═══════════════════════════════════════════════════════════════════════════════
def suf_sem_sp(df, cerv, sem, reg_list):
    sub = df[(df["Cerveja"]==cerv) & (df["Semana"]==sem)].set_index("Regiao")
    vals = []
    for r in reg_list:
        if r not in sub.index:
            vals.append(0); continue
        row      = sub.loc[r]
        ei       = float(row.get("EI", 0) or 0)
        prod_ne  = float(row.get("Prod_NE", 0) or 0)
        t_cabo   = float(row.get("Transi_Cabo", 0) or 0)
        dem      = float(row.get("Demanda", 1) or 1)
        dem_prox = float(row.get("Dem_Proxima", dem) or dem)
        ef       = ei + prod_ne + t_cabo - dem
        suf      = round(ef / (dem_prox / 6), 1) if dem_prox > 0 else 0
        vals.append(suf)
    return vals

def suf_antes(df, cerv, sem, reg_list):
    sub = df[(df["Cerveja"]==cerv) & (df["Semana"]==sem)].set_index("Regiao")
    vals = []
    for r in reg_list:
        if r not in sub.index:
            vals.append(0); continue
        row      = sub.loc[r]
        ef_orig  = pd.to_numeric(row.get("EF_Semana", 0), errors="coerce") or 0
        dem_prox = pd.to_numeric(row.get("Dem_Proxima", row.get("Demanda", 1)), errors="coerce") or 1
        vals.append(round(float(ef_orig) / (float(dem_prox) / 6), 1))
    return vals

def get_vol(df, cerv, sem, reg_list, col):
    sub = df[(df["Cerveja"]==cerv) & (df["Semana"]==sem)].set_index("Regiao")
    return [round(float(sub.loc[r,col]),1) if r in sub.index else 0 for r in reg_list]

# ── Monta os 4 cenários ───────────────────────────────────────────────────────
# A: Antes (sem redistribuição)          → df_all,   suf_antes
# B: Redist base (sem SP)                → df_final, suf_sem_sp
# C: Redist otim (sem SP)                → df_otim,  suf_sem_sp
# D: Redist base (com SP) já calculado   → df_final, Suf_f
# E: Redist otim (com SP) já calculado   → df_otim,  Suf_f

cenarios_suf = {
    "Antes":               {"df": df_all,   "fn": "antes"},
    "Redist base (s/ SP)": {"df": df_final, "fn": "sem_sp"},
    "Redist otim (s/ SP)": {"df": df_otim,  "fn": "sem_sp"},
}
cenarios_vol = {
    "Redist base":         {"df": df_final},
    "Redist otim":         {"df": df_otim},
}

CORES_SUF = {
    "Antes":               "#B4B2A9",
    "Redist base (s/ SP)": "#378ADD",
    "Redist otim (s/ SP)": "#1D9E75",
}
CORES_VOL = {
    "Redist base": "#378ADD",
    "Redist otim": "#1D9E75",
}

# Preenche dicts
suf_d, vsp_d, vne_d = {}, {}, {}
for label, cfg in cenarios_suf.items():
    suf_d[label] = {}
    for c in cervejas:
        key = c.strip()
        suf_d[label][key] = {}
        for s in semanas:
            if cfg["fn"] == "antes":
                suf_d[label][key][s] = suf_antes(cfg["df"], c, s, regioes)
            else:
                suf_d[label][key][s] = suf_sem_sp(cfg["df"], c, s, regioes)

for label, cfg in cenarios_vol.items():
    vsp_d[label] = {}
    vne_d[label] = {}
    for c in cervejas:
        key = c.strip()
        vsp_d[label][key] = {}
        vne_d[label][key] = {}
        for s in semanas:
            vsp_d[label][key][s] = get_vol(cfg["df"], c, s, regioes, "Vem_SP")
            vne_d[label][key][s] = get_vol(cfg["df"], c, s, regioes, "Prod_NE")

# DataFrames longos
rows_suf = []
for label in cenarios_suf:
    for c in cervejas_k:
        for s in semanas:
            for ri, r in enumerate(regioes):
                rows_suf.append({"cenario": label, "cerveja": c,
                                  "semana": s, "regiao": r,
                                  "suf": suf_d[label][c][s][ri]})
df_suf = pd.DataFrame(rows_suf)

rows_vol = []
for label in cenarios_vol:
    for c in cervejas_k:
        for s in semanas:
            for ri, r in enumerate(regioes):
                rows_vol.append({"cenario": label, "cerveja": c,
                                  "semana": s, "regiao": r,
                                  "vem_sp": vsp_d[label][c][s][ri],
                                  "prod_ne": vne_d[label][c][s][ri]})
df_vol = pd.DataFrame(rows_vol)

CEN_SUF = list(cenarios_suf.keys())
CEN_VOL = list(cenarios_vol.keys())

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 1 — Histograma: distribuição de suf — efeito puro redistribuição
# ═══════════════════════════════════════════════════════════════════════════════
bins       = [0,4,8,10,12,16,40]
labels_bin = ["0–4","4–8","8–10","10–12","12–16","16+"]
cores_hist = ["#E24B4A","#F09595","#FAC775","#FAC775","#5DCAA5","#1D9E75"]

def hist_counts(df_sub):
    return [int(((df_sub["suf"]>=bins[i])&(df_sub["suf"]<bins[i+1])).sum())
            for i in range(len(bins)-1)]

fig1 = make_subplots(rows=1, cols=3, subplot_titles=CEN_SUF)
for ci, label in enumerate(CEN_SUF, start=1):
    counts = hist_counts(df_suf[df_suf["cenario"]==label])
    fig1.add_trace(go.Bar(x=labels_bin, y=counts, marker_color=cores_hist,
                          text=counts, textposition="outside",
                          showlegend=False), row=1, col=ci)
fig1.update_yaxes(title_text="Nº obs", col=1)
fig1.update_xaxes(title_text="Dias de suficiência")
fig1.update_layout(**LAYOUT,
                   title="Efeito PURO da redistribuição — distribuição de suficiência (sem SP)",
                   height=440)
fig1.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 2 — Suf média por região — 3 cenários (sem SP)
# ═══════════════════════════════════════════════════════════════════════════════
fig2 = go.Figure()
for label in CEN_SUF:
    avg = df_suf[df_suf["cenario"]==label].groupby("regiao")["suf"].mean().reindex(regioes)
    fig2.add_trace(go.Bar(name=label, x=regioes, y=avg.round(1),
                          marker_color=CORES_SUF[label],
                          text=avg.round(1), textposition="outside"))
fig2.add_hline(y=12, line_dash="dash", line_color="#E24B4A",
               annotation_text="meta 12d", annotation_position="top right")
fig2.update_layout(**LAYOUT,
                   title="Suficiência média por região — efeito puro da redistribuição (sem SP)",
                   barmode="group", height=460, yaxis_title="Dias",
                   yaxis_range=[0, df_suf["suf"].max() * 1.25],
                   legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig2.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 3 — Regiões abaixo da meta por semana (sem SP)
# ═══════════════════════════════════════════════════════════════════════════════
fig3 = go.Figure()
for label in CEN_SUF:
    sub   = df_suf[df_suf["cenario"]==label]
    below = sub.groupby("semana")["suf"].apply(lambda x:(x<12).sum()).reindex(semanas)
    fig3.add_trace(go.Bar(name=label, x=semanas, y=below,
                          marker_color=CORES_SUF[label],
                          text=below, textposition="outside"))
fig3.update_layout(**LAYOUT,
                   title="Regiões abaixo da meta (suf < 12d) por semana — sem SP",
                   barmode="group", height=440,
                   yaxis_title="Nº região × cerveja", xaxis_title="Semana",
                   legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig3.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Linhas: suf por cerveja e semana (sem SP) — 3 cenários
# ═══════════════════════════════════════════════════════════════════════════════
fig4 = make_subplots(rows=1, cols=len(cervejas_k),
                     subplot_titles=cervejas_k, shared_yaxes=True)
for ci, c in enumerate(cervejas_k, start=1):
    for label in CEN_SUF:
        sub     = df_suf[(df_suf["cenario"]==label)&(df_suf["cerveja"]==c)]
        avg_sem = sub.groupby("semana")["suf"].mean().reindex(semanas)
        fig4.add_trace(go.Scatter(
            x=semanas, y=avg_sem.round(1),
            name=label if ci==1 else None,
            mode="lines+markers",
            line=dict(color=CORES_SUF[label], width=2),
            marker=dict(size=6),
            showlegend=(ci==1)
        ), row=1, col=ci)
    fig4.add_hline(y=12, line_dash="dash", line_color="#E24B4A",
                   line_width=1, row=1, col=ci)
fig4.update_layout(**LAYOUT,
                   title="Suficiência por cerveja e semana — efeito puro redistribuição (sem SP)",
                   height=420, yaxis_title="Dias", yaxis_range=[0, 30],
                   legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig4.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 5/6/7 — Heatmaps suf (sem SP): antes, redist base, redist otim
# ═══════════════════════════════════════════════════════════════════════════════
eixo_y = [f"{c} | {r}" for c in cervejas_k for r in regioes]
for label in CEN_SUF:
    z = [[suf_d[label][c][s][ri] for s in semanas]
         for c in cervejas_k for ri, r in enumerate(regioes)]
    fig = go.Figure(go.Heatmap(
        z=z, x=semanas, y=eixo_y,
        colorscale=COLORSCALE, zmid=12, zmin=0, zmax=24,
        text=[[f"{v:.1f}d" for v in row] for row in z],
        texttemplate="%{text}",
        colorbar=dict(title="dias", tickvals=[0,6,12,18,24]),
    ))
    fig.update_layout(**LAYOUT,
                      title=f"Heatmap suficiência (sem SP) — {label}",
                      height=600, xaxis_title="Semana", margin=dict(l=200))
    fig.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 8 — Volume NE vs SP: prod base vs prod otim
# ═══════════════════════════════════════════════════════════════════════════════
fig8 = make_subplots(
    rows=2, cols=len(cervejas_k),
    subplot_titles=[f"{c} — base" for c in cervejas_k] +
                   [f"{c} — otim" for c in cervejas_k],
    shared_yaxes=True, vertical_spacing=0.14,
)
for ci, c in enumerate(cervejas_k, start=1):
    for label, row_n in [("Redist base", 1), ("Redist otim", 2)]:
        ne_tot = [sum(vne_d[label][c][s]) for s in semanas]
        sp_tot = [sum(vsp_d[label][c][s]) for s in semanas]
        sl = (ci==1 and row_n==1)
        fig8.add_trace(go.Bar(name="Prod NE", x=semanas, y=ne_tot,
                              marker_color="#378ADD",
                              text=[f"{v:,.0f}" for v in ne_tot],
                              textposition="inside", showlegend=sl),
                       row=row_n, col=ci)
        fig8.add_trace(go.Bar(name="Vem SP", x=semanas, y=sp_tot,
                              marker_color="#F09595",
                              text=[f"{v:,.0f}" for v in sp_tot],
                              textposition="inside", showlegend=sl),
                       row=row_n, col=ci)
fig8.update_layout(**LAYOUT,
                   title="Volume NE vs SP — prod base (linha 1) vs prod otim (linha 2)",
                   barmode="stack", height=640,
                   legend=dict(orientation="h", y=1.04, x=1, xanchor="right"))
fig8.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 9 — Volume SP: base vs otim por cerveja
# ═══════════════════════════════════════════════════════════════════════════════
fig9 = go.Figure()
for label in CEN_VOL:
    sp_tot = df_vol[df_vol["cenario"]==label].groupby("cerveja")["vem_sp"].sum().reindex(cervejas_k)
    fig9.add_trace(go.Bar(name=label, x=cervejas_k, y=sp_tot,
                          marker_color=CORES_VOL[label],
                          text=[f"{v:,.0f}" for v in sp_tot], textposition="outside"))
fig9.update_layout(**LAYOUT,
                   title="Volume total vindo de SP por cerveja — base vs otim (hl)",
                   barmode="group", height=440, yaxis_title="hl",
                   legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig9.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 10 — Heatmap volume SP: base vs otim
# ═══════════════════════════════════════════════════════════════════════════════
for label in CEN_VOL:
    z_sp = [[vsp_d[label][c][s][ri] for s in semanas]
            for c in cervejas_k for ri, r in enumerate(regioes)]
    fig = go.Figure(go.Heatmap(
        z=z_sp, x=semanas, y=eixo_y,
        colorscale=[[0,"#f5f5f3"],[0.01,"#FAC775"],[0.3,"#F09595"],[1.0,"#E24B4A"]],
        zmin=0,
        text=[[f"{v:,.0f}" if v>0 else "–" for v in row] for row in z_sp],
        texttemplate="%{text}",
        colorbar=dict(title="hl de SP"),
    ))
    fig.update_layout(**LAYOUT,
                      title=f"Volume a trazer de SP — {label}",
                      height=600, xaxis_title="Semana", margin=dict(l=200))
    fig.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 11 — Economia em volume e R$ com a maximização da produção
# ═══════════════════════════════════════════════════════════════════════════════
eco_rows = []
for c in cervejas_k:
    if c not in custos: continue
    for s in semanas:
        modal    = "Rodoviário" if s in ["W0","W1","W2"] else "Cabotagem"
        frete_hl = custos[c]["frete_rodo"] if modal=="Rodoviário" else custos[c]["frete_cabo"]
        custo_sp_hl = custo_entregue(c, modal, "SP")

        sp_base = sum(vsp_d["Redist base"][c][s])
        sp_otim = sum(vsp_d["Redist otim"][c][s])
        delta_vol = sp_base - sp_otim

        eco_rows.append({
            "cerveja":      c,
            "semana":       s,
            "sp_base_hl":   sp_base,
            "sp_otim_hl":   sp_otim,
            "delta_vol_hl": delta_vol,
            "eco_frete_R$": delta_vol * frete_hl,
            "eco_total_R$": delta_vol * custo_sp_hl,
        })
df_eco = pd.DataFrame(eco_rows)

# Volume economizado
fig11a = go.Figure()
eco_vol = df_eco.groupby("cerveja")["delta_vol_hl"].sum().reindex(cervejas_k)
fig11a.add_trace(go.Bar(x=cervejas_k, y=eco_vol,
                        marker_color=[COR_CERV[c] for c in cervejas_k],
                        text=[f"{v:,.0f} hl" for v in eco_vol],
                        textposition="outside"))
fig11a.update_layout(**LAYOUT,
                     title="Volume deixado de trazer de SP com maximização da produção (hl)",
                     height=420, yaxis_title="hl", showlegend=False)
fig11a.show()

# Economia em R$
fig11b = go.Figure()
eco_rs  = df_eco.groupby("cerveja")["eco_total_R$"].sum().reindex(cervejas_k) / 1e3
fretes  = df_eco.groupby("cerveja")["eco_frete_R$"].sum().reindex(cervejas_k) / 1e3
fig11b.add_trace(go.Bar(name="Economia total (prod+frete)",
                        x=cervejas_k, y=eco_rs.round(1),
                        marker_color="#1D9E75",
                        text=[f"R${v:.1f}k" for v in eco_rs], textposition="outside"))
fig11b.add_trace(go.Bar(name="Só frete evitado",
                        x=cervejas_k, y=fretes.round(1),
                        marker_color="#5DCAA5",
                        text=[f"R${v:.1f}k" for v in fretes], textposition="outside"))
fig11b.update_layout(**LAYOUT,
                     title="Economia gerada pela maximização da produção NE (R$ mil)",
                     barmode="group", height=440, yaxis_title="R$ mil",
                     legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig11b.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 12 — Waterfall: decomposição da economia total
# ═══════════════════════════════════════════════════════════════════════════════
total_base = df_eco["sp_base_hl"].sum()
total_otim = df_eco["sp_otim_hl"].sum()
total_eco_vol = total_base - total_otim
total_eco_rs  = df_eco["eco_total_R$"].sum() / 1e3
total_eco_fr  = df_eco["eco_frete_R$"].sum() / 1e3
custo_prod_ec = total_eco_rs - total_eco_fr

fig12 = make_subplots(rows=1, cols=2,
                      subplot_titles=["Economia em volume (hl)", "Economia em custo (R$ mil)"])

fig12.add_trace(go.Waterfall(
    measure=["absolute","relative","total"],
    x=["SP base","(-) otimização","SP otim"],
    y=[total_base, -total_eco_vol, None],
    text=[f"{total_base:,.0f}", f"-{total_eco_vol:,.0f}", f"{total_otim:,.0f}"],
    textposition="outside",
    connector=dict(line=dict(color="#ccc")),
    increasing=dict(marker=dict(color="#F09595")),
    decreasing=dict(marker=dict(color="#1D9E75")),
    totals=dict(marker=dict(color="#378ADD")),
    showlegend=False,
), row=1, col=1)

fig12.add_trace(go.Waterfall(
    measure=["absolute","relative","relative","total"],
    x=["Custo base","(-) frete evitado","(-) prod evitada","Custo otim"],
    y=[total_base * sum(custos[c]["frete_rodo"] + custos[c]["custo_prod"]
                        for c in cervejas_k if c in custos) / len(cervejas_k) / 1e3,
       -total_eco_fr, -custo_prod_ec, None],
    text=[f"R${v:.1f}k" for v in [
        total_base * sum(custos[c]["frete_rodo"] + custos[c]["custo_prod"]
                         for c in cervejas_k if c in custos) / len(cervejas_k) / 1e3,
        total_eco_fr, custo_prod_ec,
        (total_base * sum(custos[c]["frete_rodo"] + custos[c]["custo_prod"]
                          for c in cervejas_k if c in custos) / len(cervejas_k) / 1e3) - total_eco_rs
    ]],
    textposition="outside",
    connector=dict(line=dict(color="#ccc")),
    increasing=dict(marker=dict(color="#F09595")),
    decreasing=dict(marker=dict(color="#1D9E75")),
    totals=dict(marker=dict(color="#378ADD")),
    showlegend=False,
), row=1, col=2)

fig12.update_layout(**LAYOUT,
                    title="Waterfall — impacto da maximização da produção NE",
                    height=500)
fig12.show()

# ── Resumo final ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("RESUMO DA ECONOMIA — MAXIMIZAÇÃO DA PRODUÇÃO NE")
print("="*60)
print(f"  Volume SP base:      {total_base:>10,.0f} hl")
print(f"  Volume SP otim:      {total_otim:>10,.0f} hl")
print(f"  Volume economizado:  {total_eco_vol:>10,.0f} hl  ({total_eco_vol/total_base*100:.1f}%)")
print(f"  Economia total:      R${total_eco_rs:>9,.1f} mil")
print(f"  Só frete evitado:    R${total_eco_fr:>9,.1f} mil")
print(f"  Prod evitada (custo):R${custo_prod_ec:>9,.1f} mil")
print("="*60)


RESUMO DA ECONOMIA — MAXIMIZAÇÃO DA PRODUÇÃO NE
  Volume SP base:          37,109 hl
  Volume SP otim:          39,772 hl
  Volume economizado:      -2,663 hl  (-7.2%)
  Economia total:      R$   -602.0 mil
  Só frete evitado:    R$   -208.0 mil
  Prod evitada (custo):R$   -394.0 mil


# Data frame com cenario malzbier semanda normal - sem 30%

In [12]:
FILE_PATH_2 = r"C:\Users\ferna\Downloads\AMBEV\separado.por.cerveja.xlsb"

def ler_aba_v2(file_path, sheet_name):
    rows = []
    with open_workbook(file_path) as wb:
        with wb.get_sheet(sheet_name) as sheet:
            for row in sheet.rows():
                rows.append([c.v for c in row])
    raw = pd.DataFrame(rows)
    header_row_idx = None
    for i, row in raw.iterrows():
        if row.astype(str).str.strip().str.lower().eq("demanda").any():
            header_row_idx = i
            break
    header = raw.iloc[header_row_idx].tolist()
    data   = raw.iloc[header_row_idx + 1:].reset_index(drop=True)
    semana_starts = [i for i, h in enumerate(header) if str(h).strip().lower() == "demanda"]
    COL_REGIAO    = next(i for i, h in enumerate(header) if str(h).strip() not in ["", "None", "nan"])
    records = []
    for idx_semana, col_start in enumerate(semana_starts):
        next_start = semana_starts[idx_semana + 1] if idx_semana + 1 < len(semana_starts) else None
        bloco = get_bloco_dict(header, col_start, next_start)
        def get_col(row, nomes):
            for n in nomes:
                if n in bloco:
                    return pd.to_numeric(row.iloc[bloco[n]], errors="coerce")
            return None
        for _, row in data.iterrows():
            regiao = row.iloc[COL_REGIAO]
            if pd.isna(regiao) or str(regiao).strip() in ["", "None", "TOTAL"]:
                continue
            records.append({
                "Cerveja":       sheet_name,
                "Regiao":        str(regiao).strip(),
                "Semana":        f"W{idx_semana}",
                "Demanda":       get_col(row, ["Demanda"]),
                "WSNP":          get_col(row, ["WSNP", "Malha", "0.0"]),
                "EI_Semana":     get_col(row, ["EI Semana"]),
                "Suf_in":        get_col(row, ["Suf.ini(d)"]),
                "Transi_Intern": get_col(row, ["Transf. Interna"]),
                "Transi_Cabo":   get_col(row, ["Transf. Ext (Cabo)"]),
                "Transi_Rodo":   get_col(row, ["Transf. Ext (Rodo)"]),
                "Transito":      get_col(row, ["Trânsito"]),
                "EF_Semana":     get_col(row, ["EF Semana"]),
                "Suf_f":         get_col(row, ["Suf. f (d)"]),
            })
    return pd.DataFrame(records)

# ── Lê a aba ──────────────────────────────────────────────────────────────────
df_malz30 = ler_aba_v2(FILE_PATH_2, "malzbier antes 30")

# ── Ajusta EI/Suf só em W0 ───────────────────────────────────────────────────
ei_w0 = df_malz30[df_malz30["Semana"]=="W0"][["Regiao","EI_Semana","Suf_in"]].copy()
df_malz30 = df_malz30.drop(columns=["EI_Semana","Suf_in"])
df_malz30 = df_malz30.merge(ei_w0, on="Regiao", how="left")
df_malz30.loc[df_malz30["Semana"]!="W0", ["EI_Semana","Suf_in"]] = None

# ── Visualiza ─────────────────────────────────────────────────────────────────
for sem, grupo in df_malz30.groupby("Semana"):
    print(f"\n{'='*50}  {sem}  {'='*50}")
    display(grupo.reset_index(drop=True))


==================================================  W0  ==================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f,EI_Semana,Suf_in
0,malzbier antes 30,Mapapi,W0,4672.926,0.0,5050.04832,0.0,0.0,0.0,2362.73832,2.931619,1985.616,2.549515
1,malzbier antes 30,NE Norte,W0,1516.734,16200.0,-9281.16846,0.0,0.0,0.0,5704.10154,19.636340,302.004,1.194688
2,malzbier antes 30,NE Sul,W0,2484.882,0.0,3054.15054,0.0,0.0,0.0,4952.26854,10.981248,4383.000,10.583199
3,malzbier antes 30,NO Araguaia,W0,60.660,0.0,60.65568,0.0,0.0,0.0,-0.00432,-0.000363,0.000,0.000000
4,malzbier antes 30,NO Centro,W0,1713.474,0.0,1116.53520,0.0,0.0,0.0,367.87920,1.164986,964.818,3.378463



==================================================  W1  ==================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f,EI_Semana,Suf_in
0,malzbier antes 30,Mapapi,W1,4835.700,0.0,6161.58468,0.0,0.0,0.0,3688.62300,6.756945,NaN,NaN
1,malzbier antes 30,NE Norte,W1,1742.922,0.0,-723.14280,0.0,0.0,0.0,3238.03674,14.788189,NaN,NaN
2,malzbier antes 30,NE Sul,W1,2705.850,0.0,316.95048,0.0,0.0,0.0,2563.36902,7.722503,NaN,NaN
3,malzbier antes 30,NO Araguaia,W1,71.442,0.0,71.43354,0.0,0.0,0.0,-0.01278,-0.001584,NaN,NaN
4,malzbier antes 30,NO Centro,W1,1894.680,9000.0,-5826.65288,0.0,0.0,77.0,1723.54632,7.512068,NaN,NaN



==================================================  W2  ==================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f,EI_Semana,Suf_in
0,malzbier antes 30,Mapapi,W2,3275.406,0.0,2792.98818,0.0,0.0,0.0,3206.20518,4.805074,NaN,NaN
1,malzbier antes 30,NE Norte,W2,1313.766,12960.0,-10468.70838,0.0,0.0,0.0,4415.56236,18.677641,NaN,NaN
2,malzbier antes 30,NE Sul,W2,1991.610,0.0,7662.10268,0.0,0.0,0.0,8233.86170,22.477544,NaN,NaN
3,malzbier antes 30,NO Araguaia,W2,48.402,0.0,48.39048,0.0,0.0,0.0,-0.02430,-0.002876,NaN,NaN
4,malzbier antes 30,NO Centro,W2,1376.622,7560.0,-34.37550,0.0,0.0,52.0,7924.54882,30.510248,NaN,NaN



==================================================  W3  ==================================================


,Cerveja,Regiao,Semana,Demanda,WSNP,Transi_Intern,Transi_Cabo,Transi_Rodo,Transito,EF_Semana,Suf_f,EI_Semana,Suf_in
0,malzbier antes 30,Mapapi,W3,4003.524,0.0,3048.11046,0.0,0.0,0.0,2250.79164,3.373216,NaN,NaN
1,malzbier antes 30,NE Norte,W3,1418.454,0.0,-1899.17622,0.0,0.0,0.0,1097.93214,4.644206,NaN,NaN
2,malzbier antes 30,NE Sul,W3,2197.890,0.0,1441.71306,0.0,0.0,0.0,7477.68476,20.413264,NaN,NaN
3,malzbier antes 30,NO Araguaia,W3,50.688,0.0,50.69484,0.0,0.0,0.0,-0.01746,-0.002067,NaN,NaN
4,malzbier antes 30,NO Centro,W3,1558.404,0.0,-2641.25484,0.0,0.0,0.0,3724.88998,14.341172,NaN,NaN


# Comparativo de suficiencia antes 30 e depois ( com e sem redistribuicao e maximizacao)

In [13]:
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

pio.renderers.default = "vscode"

regioes  = ["Mapapi", "NE Norte", "NE Sul", "NO Araguaia", "NO Centro"]
semanas  = ["W0", "W1", "W2", "W3"]

def get_suf(df, cerv_filter, sem, reg_list, col_suf="Suf_f"):
    sub = df[df["Semana"]==sem].copy()
    if cerv_filter:
        sub = sub[sub["Cerveja"].str.strip()==cerv_filter.strip()]
    sub = sub.set_index("Regiao")
    return [round(float(sub.loc[r, col_suf]),1) if r in sub.index else 0 for r in reg_list]

def get_suf_antes_raw(df, cerv, sem, reg_list):
    sub = df[(df["Cerveja"]==cerv) & (df["Semana"]==sem)].set_index("Regiao")
    vals = []
    for r in reg_list:
        if r not in sub.index:
            vals.append(0); continue
        row      = sub.loc[r]
        ef_orig  = pd.to_numeric(row.get("EF_Semana", 0), errors="coerce") or 0
        dem_prox = pd.to_numeric(row.get("Dem_Proxima", row.get("Demanda", 1)), errors="coerce") or 1
        vals.append(round(float(ef_orig) / (float(dem_prox) / 6), 1))
    return vals

# ── 4 cenários em ordem narrativa ────────────────────────────────────────────
# 1. Normal sem redistribuir   → df_malz30   (base, tudo normal)
# 2. +30% sem redistribuir     → df_all      (demanda sobe, nada muda)
# 3. +30% redistribuído        → df_final    (redistribuição produção base)
# 4. +30% redistribuído + otim → df_otim     (redistribuição + prod maximizada)

CENARIOS = [
    "Normal (sem +30%)",
    "+30% sem redistribuir",
    "+30% redistribuído",
    "+30% redistribuído + otim",
]
CORES = {
    "Normal (sem +30%)":         "#B4B2A9",
    "+30% sem redistribuir":     "#E24B4A",
    "+30% redistribuído":        "#378ADD",
    "+30% redistribuído + otim": "#1D9E75",
}
LAYOUT = dict(template="plotly_white", font=dict(family="Arial", size=12))

suf = {}
for label, df_use, cerv, source in [
    ("Normal (sem +30%)",         df_malz30, None,        "suf_f"),
    ("+30% sem redistribuir",     df_all,    "malzbier ", "antes"),
    ("+30% redistribuído",        df_final,  "malzbier ", "suf_f"),
    ("+30% redistribuído + otim", df_otim,   "malzbier ", "suf_f"),
]:
    suf[label] = {}
    for s in semanas:
        if source == "antes":
            suf[label][s] = get_suf_antes_raw(df_use, "malzbier ", s, regioes)
        else:
            suf[label][s] = get_suf(df_use, cerv, s, regioes, "Suf_f")

rows = []
for label in CENARIOS:
    for s in semanas:
        for ri, r in enumerate(regioes):
            rows.append({"cenario": label, "semana": s,
                         "regiao": r, "suf": suf[label][s][ri]})
df_comp = pd.DataFrame(rows)

COLORSCALE = [[0.0,"#E24B4A"],[0.25,"#F09595"],[0.45,"#FAC775"],
              [0.5,"#FAC775"],[0.55,"#85B7EB"],[0.75,"#378ADD"],[1.0,"#1D9E75"]]

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 1 — Suficiência média por região — narrativa completa
# ═══════════════════════════════════════════════════════════════════════════════
fig1 = go.Figure()
for label in CENARIOS:
    avg = df_comp[df_comp["cenario"]==label].groupby("regiao")["suf"].mean().reindex(regioes)
    fig1.add_trace(go.Bar(
        name=label, x=regioes, y=avg.round(1),
        marker_color=CORES[label],
        text=avg.round(1), textposition="outside"
    ))
fig1.add_hline(y=12, line_dash="dash", line_color="#E24B4A",
               annotation_text="meta 12d", annotation_position="top right")
fig1.update_layout(**LAYOUT,
    title="Malzbier — suficiência média por região: do normal ao otimizado",
    barmode="group", height=480, yaxis_title="Dias",
    yaxis_range=[0, df_comp["suf"].max() * 1.25],
    legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig1.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 2 — Suficiência média por semana — narrativa completa
# ═══════════════════════════════════════════════════════════════════════════════
fig2 = go.Figure()
for label in CENARIOS:
    avg = df_comp[df_comp["cenario"]==label].groupby("semana")["suf"].mean().reindex(semanas)
    fig2.add_trace(go.Bar(
        name=label, x=semanas, y=avg.round(1),
        marker_color=CORES[label],
        text=avg.round(1), textposition="outside"
    ))
fig2.add_hline(y=12, line_dash="dash", line_color="#E24B4A",
               annotation_text="meta 12d", annotation_position="top right")
fig2.update_layout(**LAYOUT,
    title="Malzbier — suficiência média por semana: do normal ao otimizado",
    barmode="group", height=460, yaxis_title="Dias",
    yaxis_range=[0, df_comp["suf"].max() * 1.25],
    legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig2.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 3 — Regiões abaixo da meta por semana
# ═══════════════════════════════════════════════════════════════════════════════
fig3 = go.Figure()
for label in CENARIOS:
    sub   = df_comp[df_comp["cenario"]==label]
    below = sub.groupby("semana")["suf"].apply(lambda x:(x<12).sum()).reindex(semanas)
    fig3.add_trace(go.Bar(
        name=label, x=semanas, y=below,
        marker_color=CORES[label],
        text=below, textposition="outside"
    ))
fig3.update_layout(**LAYOUT,
    title="Malzbier — regiões abaixo da meta (suf < 12d) por semana",
    barmode="group", height=460,
    yaxis_title="Nº regiões", xaxis_title="Semana",
    yaxis_range=[0, len(regioes) + 1],
    legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig3.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Linhas por região: evolução semana a semana — 4 cenários
# ═══════════════════════════════════════════════════════════════════════════════
fig4 = make_subplots(rows=1, cols=len(regioes),
                     subplot_titles=regioes, shared_yaxes=True)
for ri, r in enumerate(regioes):
    for label in CENARIOS:
        vals = [suf[label][s][ri] for s in semanas]
        fig4.add_trace(go.Scatter(
            x=semanas, y=vals,
            name=label if ri==0 else None,
            mode="lines+markers",
            line=dict(color=CORES[label], width=2),
            marker=dict(size=6),
            showlegend=(ri==0)
        ), row=1, col=ri+1)
    fig4.add_hline(y=12, line_dash="dash", line_color="#E24B4A",
                   line_width=1, row=1, col=ri+1)
fig4.update_layout(**LAYOUT,
    title="Malzbier — evolução suf por região: do normal ao otimizado",
    height=420, yaxis_title="Dias", yaxis_range=[0, 30],
    legend=dict(orientation="h", y=1.12, x=1, xanchor="right"))
fig4.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 5 — Heatmaps: 1 por cenário (narrativa visual)
# ═══════════════════════════════════════════════════════════════════════════════
for label in CENARIOS:
    z = [[suf[label][s][ri] for s in semanas] for ri, r in enumerate(regioes)]
    fig = go.Figure(go.Heatmap(
        z=z, x=semanas, y=regioes,
        colorscale=COLORSCALE, zmid=12, zmin=0, zmax=24,
        text=[[f"{v:.1f}d" for v in row] for row in z],
        texttemplate="%{text}",
        colorbar=dict(title="dias", tickvals=[0,6,12,18,24]),
    ))
    fig.update_layout(**LAYOUT,
        title=f"Malzbier — heatmap suficiência: {label}",
        height=340, xaxis_title="Semana", margin=dict(l=120, r=120))
    fig.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 6 — Delta: ganho de suf cenário a cenário
# (mostra o quanto cada etapa melhora vs anterior)
# ═══════════════════════════════════════════════════════════════════════════════
pares = [
    ("+30% sem redistribuir",     "Normal (sem +30%)",         "Impacto do +30% demanda"),
    ("+30% redistribuído",        "+30% sem redistribuir",     "Ganho da redistribuição"),
    ("+30% redistribuído + otim", "+30% redistribuído",        "Ganho da prod otimizada"),
]

fig6 = make_subplots(rows=1, cols=3,
                     subplot_titles=[p[2] for p in pares],
                     shared_yaxes=True)
for ci, (label_novo, label_ref, titulo) in enumerate(pares, start=1):
    delta = []
    for r in regioes:
        avg_novo = df_comp[(df_comp["cenario"]==label_novo)&(df_comp["regiao"]==r)]["suf"].mean()
        avg_ref  = df_comp[(df_comp["cenario"]==label_ref) &(df_comp["regiao"]==r)]["suf"].mean()
        delta.append(round(avg_novo - avg_ref, 1))
    cores_delta = ["#1D9E75" if v >= 0 else "#E24B4A" for v in delta]
    fig6.add_trace(go.Bar(
        x=regioes, y=delta,
        marker_color=cores_delta,
        text=[f"{v:+.1f}d" for v in delta],
        textposition="outside",
        showlegend=False
    ), row=1, col=ci)
    fig6.add_hline(y=0, line_color="#333", line_width=0.8, row=1, col=ci)

fig6.update_layout(**LAYOUT,
    title="Malzbier — delta de suficiência entre cenários (dias)",
    height=460, yaxis_title="Δ dias",
    margin=dict(t=80))
fig6.show()

# ═══════════════════════════════════════════════════════════════════════════════
# FIG 7 — Scorecard: % de regiões×semanas na meta por cenário
# ═══════════════════════════════════════════════════════════════════════════════
total_obs = len(regioes) * len(semanas)
pct_meta  = []
for label in CENARIOS:
    sub  = df_comp[df_comp["cenario"]==label]
    pct  = round((sub["suf"] >= 12).sum() / len(sub) * 100, 1)
    pct_meta.append(pct)

fig7 = go.Figure(go.Bar(
    x=CENARIOS, y=pct_meta,
    marker_color=[CORES[l] for l in CENARIOS],
    text=[f"{v}%" for v in pct_meta],
    textposition="outside"
))
fig7.add_hline(y=100, line_dash="dash", line_color="#1D9E75",
               annotation_text="100% na meta", annotation_position="top right")
fig7.update_layout(**LAYOUT,
    title="Malzbier — % de regiões×semanas atingindo a meta (suf ≥ 12d)",
    height=440, yaxis_title="%", yaxis_range=[0, 115],
    xaxis_tickangle=-15)
fig7.show()